<a href="https://colab.research.google.com/github/johanndeboda/AAI2026/blob/2026fall/ML/house_price_prediction.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.metrics import r2_score, mean_absolute_error

# Load the Ames Housing dataset from the repo
# Data source: Ames Housing dataset (Kaggle)
# https://www.kaggle.com/datasets/lespin/house-prices-dataset
# 1,460 residential property sales in Ames, Iowa, 2006-2010
url = 'https://raw.githubusercontent.com/johanndeboda/AAI2026/refs/heads/2026fall/ML/house_price.csv'
df = pd.read_csv(url)
df = df[['GrLivArea', 'Neighborhood', 'SalePrice']]

print(f"Loaded {len(df)} rows, {df['Neighborhood'].nunique()} neighborhoods")
print(df.head(), "\n")

# Features and target
X = df[['GrLivArea', 'Neighborhood']]
y = df['SalePrice']

# Preprocessing: One-hot encode the Neighborhood column
preprocessor = ColumnTransformer(
    transformers=[
        ('location', OneHotEncoder(sparse_output=False), ['Neighborhood'])
    ],
    remainder='passthrough'
)

# Create pipeline with preprocessing and model
model = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('regressor', LinearRegression())
])

# Split data
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

# Train model
model.fit(X_train, y_train)

# Evaluate on the held-out test set
y_pred = model.predict(X_test)
print(f"R-squared on test set: {r2_score(y_test, y_pred):.3f}")
print(f"Mean absolute error: ${mean_absolute_error(y_test, y_pred):,.2f}\n")

# Make prediction for a new house: 2000 sq ft in North Ames
new_house = pd.DataFrame({'GrLivArea': [2000], 'Neighborhood': ['NAmes']})
predicted_price = model.predict(new_house)
print(f"Predicted price for a 2000 sq ft house in NAmes: ${predicted_price[0]:,.2f}")

# Display model coefficients
feature_names = (
    model.named_steps['preprocessor']
    .named_transformers_['location']
    .get_feature_names_out(['Neighborhood'])
).tolist() + ['GrLivArea']

coefficients = model.named_steps['regressor'].coef_

print("\nModel Coefficients:")
for feature, coef in zip(feature_names, coefficients):
    print(f"{feature}: {coef:.2f}")

Loaded 1460 rows, 25 neighborhoods
   GrLivArea Neighborhood  SalePrice
0       1710      CollgCr     208500
1       1262      Veenker     181500
2       1786      CollgCr     223500
3       1717      Crawfor     140000
4       2198      NoRidge     250000 

R-squared on test set: 0.757
Mean absolute error: $28,298.36

Predicted price for a 2000 sq ft house in NAmes: $199,439.21

Model Coefficients:
Neighborhood_Blmngtn: 15916.57
Neighborhood_Blueste: -36576.67
Neighborhood_BrDale: -51780.44
Neighborhood_BrkSide: -34350.03
Neighborhood_ClearCr: 16301.31
Neighborhood_CollgCr: 19492.68
Neighborhood_Crawfor: 7062.28
Neighborhood_Edwards: -40751.05
Neighborhood_Gilbert: 2269.27
Neighborhood_IDOTRR: -49248.61
Neighborhood_MeadowV: -52983.05
Neighborhood_Mitchel: -9646.37
Neighborhood_NAmes: -20084.98
Neighborhood_NPkVill: -17811.23
Neighborhood_NWAmes: -9598.79
Neighborhood_NoRidge: 70532.32
Neighborhood_NridgHt: 97818.13
Neighborhood_OldTown: -53131.10
Neighborhood_SWISU: -62843.19
Neighbo


## What the square footage coefficient means
The square footage coefficient means about how much more a house will sell based on how much more square feet bigger it is than the point of comparison. In this case, the coefficient is $76.64 meaning an increase in 1 square foot is about that much more in terms of selling price before the increase or decrease per neighborhood.

## How location affects price
The location affects price as some neighborhoods are more expensive meaning that although the base coefficient is 76.64, a neighborhood like NAmes compared to MeadowV can have a flat rate increase which in the scenario would be over $32,000
